# 06. Also-Watched Recommender

## Objective

The goal of this notebook is to build an "also-watched" movie recommender using user-rating co-occurrence patterns.

We will:

* Load the processed ratings data
* Identify movies rated by the same users
* Measure movie-to-movie co-occurrence
* Generate "people also watched/rated" recommendations
* Test the recommender with example movies
* Save the required data for later use

This recommender will provide another behavioral signal that can be combined with the content-based and collaborative-filtering models in the hybrid recommender.


In [1]:
# Import the libraries needed to build the also-watched recommender
import pandas as pd
import numpy as np

from pathlib import Path

In [2]:
# Define the path to the processed ratings dataset
PROJECT_DIR = Path.cwd().parent
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

print("Processed data:", PROCESSED_DIR)

Processed data: c:\Users\moham\Desktop\Netflix Movie recomendation system\data\processed


### Load the rating data

In [3]:
# Load the processed ratings data used to find movie co-occurrence patterns
ratings_cf = pd.read_csv(
    PROCESSED_DIR / "ratings_cf.csv"
)

print("Ratings loaded successfully.")
print("Shape:", ratings_cf.shape)

Ratings loaded successfully.
Shape: (29542223, 4)


In [4]:
# Count how many users have rated each movie
movie_user_counts = (
    ratings_cf.groupby("movieId")["userId"]
    .nunique()
)

print("Number of movies:", len(movie_user_counts))
print("Minimum users per movie:", movie_user_counts.min())
print("Median users per movie:", movie_user_counts.median())
print("Maximum users per movie:", movie_user_counts.max())

Number of movies: 31701
Minimum users per movie: 10
Median users per movie: 50.0
Maximum users per movie: 80015


### Create a movie–user relationship table

In [5]:
# Create a compact table containing each user-movie rating relationship
user_movie_pairs = ratings_cf[
    ["userId", "movieId"]
].drop_duplicates()

print("User-movie pairs:", len(user_movie_pairs))
print(user_movie_pairs.head())

User-movie pairs: 29542223
   userId  movieId
0       1       17
1       1       25
2       1       29
3       1       30
4       1       32


### Build the movie–user sparse matrix
Now putting movies in rows and users in columns.

In [6]:
# Build a sparse movie-user matrix where each row represents a movie
from scipy.sparse import csr_matrix

movie_ids_aw = user_movie_pairs["movieId"].unique()
user_ids_aw = user_movie_pairs["userId"].unique()

movie_to_index_aw = {
    movie_id: index
    for index, movie_id in enumerate(movie_ids_aw)
}

user_to_index_aw = {
    user_id: index
    for index, user_id in enumerate(user_ids_aw)
}

movie_indices_aw = user_movie_pairs["movieId"].map(
    movie_to_index_aw
).to_numpy()

user_indices_aw = user_movie_pairs["userId"].map(
    user_to_index_aw
).to_numpy()

movie_user_matrix = csr_matrix(
    (
        np.ones(len(user_movie_pairs), dtype=np.float32),
        (movie_indices_aw, user_indices_aw)
    ),
    shape=(len(movie_ids_aw), len(user_ids_aw))
)

print("Matrix shape:", movie_user_matrix.shape)
print("Stored interactions:", movie_user_matrix.nnz)

Matrix shape: (31701, 128255)
Stored interactions: 29542223


### Calculate movie-to-movie co-occurrence

In [7]:
# Calculate how many users each pair of movies has in common
movie_cooccurrence = movie_user_matrix @ movie_user_matrix.T

print("Co-occurrence matrix shape:", movie_cooccurrence.shape)
print("Stored co-occurrence values:", movie_cooccurrence.nnz)

Co-occurrence matrix shape: (31701, 31701)
Stored co-occurrence values: 708897677


### Remove the huge co-occurrence matrix

In [8]:
# Remove the large full co-occurrence matrix from memory
del movie_cooccurrence

print("Full co-occurrence matrix removed from memory.")

Full co-occurrence matrix removed from memory.


### Create a function to find movies rated by the same users

For a selected movie, we'll:

- Find all users who rated it.
- Find every other movie those users rated.
- Count how many times each other movie appears.

In [9]:
# Create a function that counts movies rated by the same users as a selected movie
def get_cooccurrence_counts(movie_id):

    # Find the matrix index of the selected movie
    movie_index = movie_to_index_aw.get(movie_id)

    # Check whether the movie exists
    if movie_index is None:
        return None

    # Get the users who rated the selected movie
    user_indices = movie_user_matrix[movie_index].indices

    # Find all movies rated by those users
    related_movies_matrix = movie_user_matrix[:, user_indices]

    # Count how many of those users rated each movie
    cooccurrence_counts = np.asarray(
        related_movies_matrix.sum(axis=1)
    ).ravel()

    # Remove the selected movie itself
    cooccurrence_counts[movie_index] = 0

    return cooccurrence_counts

In [10]:
# Calculate co-occurrence counts for The Matrix
matrix_cooccurrence = get_cooccurrence_counts(2571)

print("Number of movie counts:", len(matrix_cooccurrence))
print("Highest co-occurrence:", matrix_cooccurrence.max())

Number of movie counts: 31701
Highest co-occurrence: 53281.0


### Convert Co-Occurrence into a Normalized Similarity

We'll use **cosine similarity**:

$$
\text{similarity}(A,B)
=
\frac{\text{common users}}
{\sqrt{\text{users of A}\times\text{users of B}}}
$$

### Why this is useful

Suppose:

* Matrix → 50,000 users
* Movie B → 60,000 users
* Common users → 30,000

Then:

$$
\frac{30,000}{\sqrt{50,000\times60,000}}
\approx 0.548
$$

The score is now between **0 and 1**:

* `1` → identical user-rating pattern
* `0` → no users in common
* Higher score → stronger similarity in user behavior

This is a much better basis for our **"people also watched/rated"** signal.


In [11]:
# Convert Matrix's co-occurrence counts into cosine similarity scores
matrix_user_count = movie_user_counts.loc[2571]

matrix_similarity = (
    matrix_cooccurrence
    / np.sqrt(
        matrix_user_count
        * movie_user_counts.reindex(movie_ids_aw).to_numpy()
    )
)

matrix_similarity[~np.isfinite(matrix_similarity)] = 0

print("Highest similarity:", round(matrix_similarity.max(), 4))

Highest similarity: 0.7496


### Find the top also-watched movies

In [13]:
# Load movie metadata so we can display movie titles in the recommendations
movies_content = pd.read_csv(
    PROCESSED_DIR / "movies_content.csv"
)

print("Movie metadata loaded:", movies_content.shape)

Movie metadata loaded: (87585, 6)


In [14]:
# Find the top movies that share the strongest user-rating patterns with The Matrix
top_indices = np.argsort(matrix_similarity)[::-1][:11]

top_movie_ids = [
    movie_ids_aw[index]
    for index in top_indices
]

top_scores = matrix_similarity[top_indices]

also_watched_matrix = movies_content[
    movies_content["movieId"].isin(top_movie_ids)
][["movieId", "title"]].copy()

also_watched_matrix["similarity_score"] = (
    also_watched_matrix["movieId"]
    .map(dict(zip(top_movie_ids, top_scores)))
)

also_watched_matrix = (
    also_watched_matrix
    .set_index("movieId")
    .loc[top_movie_ids]
    .reset_index()
)

also_watched_matrix

,movieId,title,similarity_score
0,2959,Fight Club (1999),0.749632
1,4993,"Lord of the Rings: The Fellowship of the Ring,...",0.747542
2,5952,"Lord of the Rings: The Two Towers, The (2002)",0.723598
3,7153,"Lord of the Rings: The Return of the King, The...",0.720354
4,1196,Star Wars: Episode V - The Empire Strikes Back...,0.711337
5,260,Star Wars: Episode IV - A New Hope (1977),0.708945
6,3578,Gladiator (2000),0.687649
7,318,"Shawshank Redemption, The (1994)",0.686877
8,296,Pulp Fiction (1994),0.681439
9,356,Forrest Gump (1994),0.680578


### Build the reusable also-watched function

In [15]:
# Create a reusable function for generating also-watched movie recommendations
def recommend_also_watched(movie_id, n=10):

    # Get the co-occurrence counts for the selected movie
    cooccurrence_counts = get_cooccurrence_counts(movie_id)

    # Check whether the movie exists
    if cooccurrence_counts is None:
        return "Movie ID not found."

    # Get the number of users who rated the selected movie
    selected_movie_index = movie_to_index_aw[movie_id]
    selected_movie_users = movie_user_counts.loc[movie_id]

    # Calculate cosine similarity with every movie
    similarity_scores = (
        cooccurrence_counts
        / np.sqrt(
            selected_movie_users
            * movie_user_counts.reindex(movie_ids_aw).to_numpy()
        )
    )

    # Replace invalid values with zero
    similarity_scores[~np.isfinite(similarity_scores)] = 0

    # Select the top unseen movies
    top_indices = np.argsort(similarity_scores)[::-1][:n]

    # Convert matrix positions to movie IDs
    recommended_movie_ids = [
        movie_ids_aw[index]
        for index in top_indices
    ]

    # Create a readable recommendation table
    recommendations = movies_content[
        movies_content["movieId"].isin(recommended_movie_ids)
    ][["movieId", "title"]].copy()

    # Add similarity scores
    score_map = dict(
        zip(
            recommended_movie_ids,
            similarity_scores[top_indices]
        )
    )

    recommendations["similarity_score"] = (
        recommendations["movieId"].map(score_map)
    )

    # Preserve the recommendation order
    recommendations = (
        recommendations
        .set_index("movieId")
        .loc[recommended_movie_ids]
        .reset_index()
    )

    return recommendations

### Test it with The Matrix

In [16]:
# Test the also-watched recommender with The Matrix
recommend_also_watched(2571, n=10)

,movieId,title,similarity_score
0,2959,Fight Club (1999),0.749632
1,4993,"Lord of the Rings: The Fellowship of the Ring,...",0.747542
2,5952,"Lord of the Rings: The Two Towers, The (2002)",0.723598
3,7153,"Lord of the Rings: The Return of the King, The...",0.720354
4,1196,Star Wars: Episode V - The Empire Strikes Back...,0.711337
5,260,Star Wars: Episode IV - A New Hope (1977),0.708945
6,3578,Gladiator (2000),0.687649
7,318,"Shawshank Redemption, The (1994)",0.686877
8,296,Pulp Fiction (1994),0.681439
9,356,Forrest Gump (1994),0.680578


### Test an invalid movie ID

In [17]:
# Test how the recommender handles a movie ID that does not exist
recommend_also_watched(9999999, n=10)

'Movie ID not found.'

### Save the movie-user matrix

We don't want the Streamlit app to rebuild the 29.5-million-interaction matrix every time it starts. We'll save the sparse matrix to the models/ folder so it can be loaded later.

In [18]:
# Save the sparse movie-user interaction matrix for reuse in the application
from scipy.sparse import save_npz

MODELS_DIR = PROJECT_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

save_npz(
    MODELS_DIR / "movie_user_matrix.npz",
    movie_user_matrix
)

print("Movie-user matrix saved successfully.")

Movie-user matrix saved successfully.


### Save the movie mapping
The matrix rows correspond to the order stored in movie_ids_aw. We need to preserve that mapping so the application knows which movie each matrix row represents.

In [19]:
# Save the movie IDs and their matrix-index mapping for later use
import joblib

np.save(
    MODELS_DIR / "movie_ids_aw.npy",
    movie_ids_aw
)

joblib.dump(
    movie_to_index_aw,
    MODELS_DIR / "movie_to_index_aw.joblib"
)

print("Also-watched movie mappings saved successfully.")

Also-watched movie mappings saved successfully.


### Save movie user counts
We also need the number of users who rated each movie because the cosine similarity calculation uses the selected movie's user count and every candidate movie's user count.

In [20]:
# Save the number of users who rated each movie for future similarity calculations
movie_user_counts.to_csv(
    MODELS_DIR / "movie_user_counts.csv",
    header=["user_count"]
)

print("Movie user counts saved successfully.")

Movie user counts saved successfully.


### Reload the saved components

In [21]:
# Reload the saved also-watched components to verify they can be reused
from scipy.sparse import load_npz

movie_user_matrix_loaded = load_npz(
    MODELS_DIR / "movie_user_matrix.npz"
)

movie_ids_aw_loaded = np.load(
    MODELS_DIR / "movie_ids_aw.npy"
)

movie_to_index_aw_loaded = joblib.load(
    MODELS_DIR / "movie_to_index_aw.joblib"
)

movie_user_counts_loaded = pd.read_csv(
    MODELS_DIR / "movie_user_counts.csv",
    index_col=0
)["user_count"]

print("Matrix shape:", movie_user_matrix_loaded.shape)
print("Movie IDs:", len(movie_ids_aw_loaded))
print("Movie mapping:", len(movie_to_index_aw_loaded))
print("Movie user counts:", len(movie_user_counts_loaded))

Matrix shape: (31701, 128255)
Movie IDs: 31701
Movie mapping: 31701
Movie user counts: 31701


## Conclusion

The also-watched recommender was successfully developed using movie-rating co-occurrence patterns.

The model:

* Builds a sparse movie-user interaction matrix
* Measures how many users rated the same movies
* Converts co-occurrence into cosine similarity scores
* Generates "people also watched/rated" recommendations
* Handles invalid movie IDs safely
* Saves the required matrix, mappings, and user-count data for later use

For example, **The Matrix** produced strong behavioral similarities with movies such as *Fight Club*, *The Lord of the Rings*, *Star Wars*, *Gladiator*, and *The Shawshank Redemption*.

The similarity scores represent **user-rating behavior**, not predicted ratings or direct movie-content similarity.

Because MovieLens contains rating activity rather than actual watch-history data, this recommender should technically be interpreted as **"people also rated"**. It can later be presented in a Netflix-style interface as an "also watched/rated" signal.

The saved components are stored in the `models/` directory and can be reused by the hybrid recommender and Streamlit application.

### Next Step

The next notebook will combine the **content-based**, **collaborative-filtering**, and **also-watched** signals into a hybrid recommendation system.
